# Task 1: Financial News EDA

This notebook covers the Task 1 deliverables for Nova Financial Solutions:
- Descriptive statistics
- Publisher analysis
- Time-series publication behavior
- Keyword and topic exploration

Dataset: `../newsData/raw_analyst_ratings.csv`

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

In [ ]:
DATA_PATH = "../newsData/raw_analyst_ratings.csv"

df = pd.read_csv(
    DATA_PATH,
    usecols=["headline", "url", "publisher", "date", "stock"],
    low_memory=False
)

df["headline"] = df["headline"].fillna("").astype(str)
df["publisher"] = df["publisher"].fillna("Unknown").astype(str)
df["stock"] = df["stock"].fillna("Unknown").astype(str)

df["date_ts"] = pd.to_datetime(df["date"], errors="coerce", utc=True)
df["date_only"] = df["date_ts"].dt.date
df["hour"] = df["date_ts"].dt.hour
df["headline_len"] = df["headline"].str.len()

df.shape

In [ ]:
summary = {
    "row_count": len(df),
    "unique_publishers": df["publisher"].nunique(),
    "unique_stocks": df["stock"].nunique(),
    "date_min": str(df["date_ts"].min()),
    "date_max": str(df["date_ts"].max()),
    "missing_dates": int(df["date_ts"].isna().sum()),
    "headline_len_mean": float(df["headline_len"].mean()),
    "headline_len_median": float(df["headline_len"].median()),
    "headline_len_std": float(df["headline_len"].std())
}
pd.Series(summary)

## 1) Descriptive Statistics: Headline Length

In [ ]:
fig, ax = plt.subplots()
sns.histplot(df["headline_len"], bins=60, kde=True, color="steelblue", ax=ax)
ax.set_title("Distribution of Headline Character Length")
ax.set_xlabel("Headline Length (characters)")
ax.set_ylabel("Article Count")
plt.show()

## 2) Publisher Analysis

In [ ]:
top_publishers = df["publisher"].value_counts().head(20)
top_publishers

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(x=top_publishers.values, y=top_publishers.index, palette="viridis", ax=ax)
ax.set_title("Top 20 Most Active Publishers")
ax.set_xlabel("Number of Articles")
ax.set_ylabel("Publisher")
plt.show()

In [ ]:
# Domain extraction for email-like publishers
pub_series = df["publisher"].str.lower().str.strip()
email_mask = pub_series.str.contains("@", regex=False)
domains = pub_series[email_mask].str.extract(r"@([a-z0-9.-]+)$", expand=False).dropna()
domains.value_counts().head(15)

## 3) Time Series Analysis of News Volume

In [ ]:
daily_counts = (
    df.dropna(subset=["date_ts"])
      .set_index("date_ts")
      .resample("D")["headline"]
      .count()
)

daily_counts.head()

In [ ]:
rolling_7d = daily_counts.rolling(7, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(14, 6))
daily_counts.plot(ax=ax, alpha=0.45, color="slategray", label="Daily Volume")
rolling_7d.plot(ax=ax, color="crimson", linewidth=2, label="7-day Moving Average")
ax.set_title("Daily Financial News Volume Over Time")
ax.set_xlabel("Date")
ax.set_ylabel("Number of Headlines")
ax.legend()
plt.show()

In [ ]:
hour_counts = df["hour"].dropna().astype(int).value_counts().sort_index()

fig, ax = plt.subplots()
sns.barplot(x=hour_counts.index, y=hour_counts.values, color="teal", ax=ax)
ax.set_title("Publication Activity by Hour (UTC)")
ax.set_xlabel("Hour of Day (UTC)")
ax.set_ylabel("Headline Count")
plt.show()

In [ ]:
z = (daily_counts - daily_counts.mean()) / daily_counts.std(ddof=0)
spikes = daily_counts[z > 2.5].sort_values(ascending=False).head(15)
spikes

## 4) Text Analysis: Keywords and Topics

In [ ]:
sample_size = min(120000, len(df))
text_sample = df["headline"].sample(sample_size, random_state=42) if len(df) > sample_size else df["headline"]

count_vec = CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=20, max_df=0.9)
X_count = count_vec.fit_transform(text_sample)
term_counts = np.asarray(X_count.sum(axis=0)).ravel()
terms = np.array(count_vec.get_feature_names_out())

top_idx = term_counts.argsort()[::-1][:30]
top_terms_df = pd.DataFrame({
    "term": terms[top_idx],
    "count": term_counts[top_idx]
})
top_terms_df.head(15)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
sns.barplot(data=top_terms_df.head(20), x="count", y="term", palette="magma", ax=ax)
ax.set_title("Top 20 Terms/Phrases in Headlines (CountVectorizer)")
ax.set_xlabel("Frequency")
ax.set_ylabel("Term")
plt.show()

In [ ]:
tfidf_vec = TfidfVectorizer(stop_words="english", ngram_range=(1, 2), min_df=20, max_df=0.9)
X_tfidf = tfidf_vec.fit_transform(text_sample)
tfidf_scores = np.asarray(X_tfidf.mean(axis=0)).ravel()
tfidf_terms = np.array(tfidf_vec.get_feature_names_out())

top_tfidf_idx = tfidf_scores.argsort()[::-1][:20]
pd.DataFrame({
    "term": tfidf_terms[top_tfidf_idx],
    "avg_tfidf": tfidf_scores[top_tfidf_idx]
})

In [ ]:
lda_vec = CountVectorizer(stop_words="english", min_df=30, max_df=0.85, max_features=2500)
X_lda = lda_vec.fit_transform(text_sample)

lda = LatentDirichletAllocation(n_components=6, random_state=42, learning_method="batch")
lda.fit(X_lda)

vocab = np.array(lda_vec.get_feature_names_out())
topic_rows = []
for i, comp in enumerate(lda.components_, start=1):
    top_words = vocab[comp.argsort()[::-1][:10]]
    topic_rows.append({
        "topic": f"Topic {i}",
        "keywords": ", ".join(top_words)
    })

topic_df = pd.DataFrame(topic_rows)
topic_df

## 5) Initial Insights and Task 1 Conclusions

1. Headline lengths are concentrated in a narrow range, suggesting a standardized newsroom style.
2. Publisher contribution is highly imbalanced: a few publishers dominate volume.
3. Daily volume shows distinct spikes, which can be linked to major market events in later analysis.
4. Publication timing clusters in specific hours, useful for event-window modeling in future tasks.
5. Top terms and topic clusters are strongly event-driven (highs/lows, earnings, guidance, ratings).

These findings prepare the project for sentiment scoring and correlation analysis in subsequent tasks.